In [67]:
# these are the libraries that you will need throughout the assignment
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.compose import ColumnTransformer

from matplotlib.colors import ListedColormap

from HW3 import *


from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

RSEED = 8

In [3]:
cleveland = pd.read_csv("cleveland.csv")  # change this
test = pd.read_csv("hungary.csv")     # change this

In [4]:
numerical_column = [ 'Age', 'RestBP','Chol','MaxHR','Oldpeak' ]
target = 'Num'
total_column = cleveland.columns.to_list()
categorical_column = []
for c in total_column:
    if c not in numerical_column and c != target:
        categorical_column.append(c)
categorical_column

['Sex', 'ChestPainType', 'FBS', 'RestECG', 'ExAng', 'Slope', 'Ca', 'Thal']

In [ ]:
(cleveland['Ca']=='?').sum()

In [10]:
def clean(cleveland):
    df_copy = cleveland.copy()
    df_copy.replace('?',np.nan,inplace=True)
    df_copy.isna().sum()
    df_copy.dtypes
    for col in df_copy.columns:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
    # Categorical Values hai yeh
    valid_values = {
            'Sex': [0, 1],
            'ChestPainType': [1, 2, 3, 4],
            'FBS': [0, 1],
            'RestECG': [0, 1, 2],
            'ExAng': [0, 1],
            'Slope': [1, 2, 3],
            'Ca': [0, 1, 2, 3],
            'Thal': [3, 6, 7],
            'Num': [0, 1, 2, 3, 4]
    }
    for col in numerical_column:
            if col in df_copy.columns:
                if col == 'Oldpeak':
                # Oldpeak can be 0, but not negative
                    df_copy.loc[df_copy[col] < 0, col] = np.nan
                elif col == 'Age':
                    # Age can't be <18 since adult patients bola hai or > 100
                    df_copy.loc[(df_copy[col] < 18) | (df_copy[col] > 100), col] = np.nan
                elif col == 'Chol':
                    # Cholesterol should be between 0 and 1000
                    df_copy.loc[(df_copy[col] <=0) | (df_copy[col] >= 600), col] = np.nan 
                elif col == 'MaxHR':
                    df_copy.loc[(df_copy[col] <=0) | (df_copy[col] >= 220), col] = np.nan  
                else:
                    # For other continuous features, 0 or negative are invalid
                    df_copy.loc[df_copy[col] <= 0, col] = np.nan
            # 
    # Categorical change
    for col in categorical_column:
        df_copy.loc[~df_copy[col].isin(valid_values[col]),col] = np.nan
    missing_values_count= df_copy.isna().sum().to_dict()
    print(missing_values_count)
    return (missing_values_count,df_copy)

In [11]:
a,cleveland_preprocessed = clean(cleveland)
a,test_preprocessed = clean(test)

{'Age': 5, 'Sex': 2, 'ChestPainType': 1, 'RestBP': 0, 'Chol': 3, 'FBS': 0, 'RestECG': 2, 'MaxHR': 3, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 1, 'Ca': 5, 'Thal': 3, 'Num': 0}
{'Age': 4, 'Sex': 1, 'ChestPainType': 0, 'RestBP': 0, 'Chol': 25, 'FBS': 8, 'RestECG': 2, 'MaxHR': 4, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 188, 'Ca': 289, 'Thal': 264, 'Num': 0}


In [ ]:
# After Data Preprocessing now missing values impute -> Scaling -> Training and predicting

In [24]:
X_cleveland = cleveland_preprocessed.drop(columns=['Num'])
y_cleveland = cleveland['Num']
X_test =  test_preprocessed.drop(columns=['Num'])
y_test = test['Num']

In [49]:
X_train, X_val, y_train, y_val =  train_test_split(
    X_cleveland, y_cleveland, test_size=0.3, random_state=42)

In [30]:
y_val.unique()

179    0
228    4
111    0
246    0
60     1
Name: Num, dtype: int64

In [50]:
y_val = [0 if c==0 else 1 for c in y_val]
y_train = [0 if c==0 else 1 for c in y_train]

In [ ]:
y_val

In [53]:
y_val = pd.DataFrame(y_val,columns=['Num'])

In [54]:
y_train = pd.DataFrame(y_train,columns=['Num'])

In [58]:
y_train['Num'].unique()

array([0, 1])

In [69]:
# Imputation handling for numerical and columns values

categorical_transformer = Pipeline([('knnImputer', KNNImputer(n_neighbors=2,weights='distance'))])
numeric_transformer =Pipeline([('imputer',SimpleImputer(missing_values=np.nan, strategy='mean'))])

In [71]:
column_tranformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_column),
        ("cat", categorical_transformer, categorical_column),
    ]
)

In [78]:
# Cmplete the imputation of all
column_tranformer.fit(X_train)

In [76]:
print(dir(column_tranformer))

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_prefix_for_feature_names_out', '_build_request_for_signature', '_call_func_on_transformers', '_columns', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_get_default_requests', '_get_doc_link', '_get_empty_routing', '_get_feature_name_out_for_transformer', '_get_metadata_request', '_get_param_names', '_get_params', '_get_params_html', '_get_remainder_cols', '_get_remainder_cols_dtype', '_hstack', '_html_repr', '_iter', '_log_messa

In [134]:
X_train_imputed = column_tranformer.transform(X_train)
X_val_imputed = column_tranformer.transform(X_val)
X_test_imputed = column_tranformer.transform(X_test)

In [135]:
X_train_imputed = pd.DataFrame(X_train_imputed, columns=numerical_column + categorical_column)
X_val_imputed   = pd.DataFrame(X_val_imputed,   columns=numerical_column + categorical_column)
X_test_imputed  = pd.DataFrame(X_test_imputed,  columns=numerical_column + categorical_column)


In [136]:
np.isnan(X_test_imputed).sum()

Age              0
RestBP           0
Chol             0
MaxHR            0
Oldpeak          0
Sex              0
ChestPainType    0
FBS              0
RestECG          0
ExAng            0
Slope            0
Ca               0
Thal             0
dtype: int64

In [87]:
# Classification Task : Single Split and Cross Validation using Decision Tree and Logistic Regression
hyperparameters_tree = {"criterion": ['gini', 'entropy'],"max_depth": [3, 5, 10]}
hyperparameters_logreg = {"penalty":['l1', 'l2'],"C":[0.1, 10],"solver":['liblinear']}


In [92]:
from sklearn.model_selection import ParameterGrid
grid_tree = ParameterGrid(hyperparameters_tree)
list(grid_tree)
grid_logreg = ParameterGrid(hyperparameters_logreg)
list(grid_logreg)

[{'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'},
 {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'},
 {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'},
 {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}]

In [137]:
X_train_imputed.columns

Index(['Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak', 'Sex', 'ChestPainType',
       'FBS', 'RestECG', 'ExAng', 'Slope', 'Ca', 'Thal'],
      dtype='object')

In [138]:
X_train_imputed.head()

,Age,RestBP,Chol,MaxHR,Oldpeak,Sex,ChestPainType,FBS,RestECG,ExAng,Slope,Ca,Thal
0,54.0,160.0,201.0,163.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,1.0,3.0
1,70.0,156.0,245.0,143.0,0.0,1.0,2.0,0.0,2.0,0.0,1.0,0.0,3.0
2,51.0,110.0,175.0,123.0,0.6,1.0,3.0,0.0,0.0,0.0,1.0,0.0,3.0
3,56.0,120.0,236.0,178.0,0.8,1.0,2.0,0.0,0.0,0.0,1.0,0.0,3.0
4,44.0,140.0,235.0,180.0,0.0,1.0,3.0,0.0,2.0,0.0,1.0,0.0,3.0


In [143]:
performance_df = pd.DataFrame(columns=['params', 'F1 scores'])

0

In [ ]:
# loop for desicion tree


from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
decision_tree = DecisionTreeClassifier()
# print(dir(decision_tree))
for i in list(grid_tree):
    a= train_and_evaluate_single_split(X_train_imputed,X_test_imputed,y_train,y_test,categorical_column,numerical_column,decision_tree,i)
    print(a)
    performance_df.loc[len(performance_df)] = a
    # decision_tree.set_params(**i)v
    # decision_tree.fit(X_train,y_train)
logisitc= LogisticRegression()
for i in list(grid_logreg):
    a= train_and_evaluate_single_split(X_train_imputed,X_test_imputed,y_train,y_test,categorical_column,numerical_column,logisitc,i)
    print(a)
    performance_df.loc[len(performance_df)] = a

{'params': {'criterion': 'gini', 'max_depth': 3}, 'F1 scores': 0.6197183098591549}
{'params': {'criterion': 'gini', 'max_depth': 5}, 'F1 scores': 0.5928853754940712}
{'params': {'criterion': 'gini', 'max_depth': 10}, 'F1 scores': 0.5882352941176471}
{'params': {'criterion': 'entropy', 'max_depth': 3}, 'F1 scores': 0.6296296296296297}
{'params': {'criterion': 'entropy', 'max_depth': 5}, 'F1 scores': 0.6031746031746031}
{'params': {'criterion': 'entropy', 'max_depth': 10}, 'F1 scores': 0.6031746031746031}
{'params': {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}, 'F1 scores': 0.7355371900826446}
{'params': {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}, 'F1 scores': 0.7381974248927039}
{'params': {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}, 'F1 scores': 0.7338709677419355}
{'params': {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}, 'F1 scores': 0.7338709677419355}


/Users/shubhamkeshari/Documents/SU/DS/dsvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/shubhamkeshari/Documents/SU/DS/dsvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/shubhamkeshari/Documents/SU/DS/dsvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/shubhamkeshari/Documents/SU/DS/dsvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: D

In [147]:
performance_df.sort_values(by="F1 scores",ascending=False)

,params,F1 scores
7,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.738197
6,"{'C': 0.1, 'penalty': 'l1', 'solver': 'libline...",0.735537
8,"{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}",0.733871
9,"{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}",0.733871
3,"{'criterion': 'entropy', 'max_depth': 3}",0.629630
0,"{'criterion': 'gini', 'max_depth': 3}",0.619718
4,"{'criterion': 'entropy', 'max_depth': 5}",0.603175
5,"{'criterion': 'entropy', 'max_depth': 10}",0.603175
1,"{'criterion': 'gini', 'max_depth': 5}",0.592885
2,"{'criterion': 'gini', 'max_depth': 10}",0.588235


In [148]:
def train_and_evaluate_single_split(X_train, X_val, y_train, y_val, cat_cols,num_cols,model, hp):
    model.set_params(**hp)
    # /Pipeline and Column 
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', model)
    ])

    # fitting values
    pipeline.fit(X_train, y_train)


    y_pred = pipeline.predict(X_val)

    f1 = f1_score(y_val, y_pred)

    return {'params': hp, 'F1 scores': f1}

In [149]:
X = pd.concat([X_train_imputed,X_val_imputed])
y = pd.concat([y_train,y_val])
# Using only X to get best parameter which we can use for predicting testing set

In [174]:
# Cross Validation Result Store Dont use Train Test Split nahi instead own function returning indexes
performance_df = pd.DataFrame(columns=['params', 'Average F1 scores'])
# Desicion tree
for i in list(grid_tree):
    a= train_and_evaluate_cross_validation(X,y,categorical_column,numerical_column,decision_tree,i)
    print(a)
    performance_df.loc[len(performance_df)] = a

# Logistic Regression
for i in list(grid_logreg):
    a= train_and_evaluate_cross_validation(X,y,categorical_column,numerical_column,logisitc,i)
    print(a)
    performance_df.loc[len(performance_df)] = a

AttributeError: 'ColumnTransformer' object has no attribute 'transformers_'

In [161]:
performance_df.sort_values(by="Average F1 scores",ascending=False)

,params,Average F1 scores
7,"{'C': 0.1, 'penalty': 'l2', 'solver': 'libline...",0.824586
6,"{'C': 0.1, 'penalty': 'l1', 'solver': 'libline...",0.799633
8,"{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}",0.797566
9,"{'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}",0.794558
0,"{'criterion': 'gini', 'max_depth': 3}",0.756409
3,"{'criterion': 'entropy', 'max_depth': 3}",0.753882
4,"{'criterion': 'entropy', 'max_depth': 5}",0.733404
2,"{'criterion': 'gini', 'max_depth': 10}",0.719686
1,"{'criterion': 'gini', 'max_depth': 5}",0.717556
5,"{'criterion': 'entropy', 'max_depth': 10}",0.705547


In [173]:
def train_and_evaluate_cross_validation(X,y,cat_cols,num_cols,model, hp):
    model.set_params(**hp)
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ])
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', model)
    ])
    average_f1_score = 0
    # K Fold use karke fit and predict and score
    skf = StratifiedKFold(n_splits=5)
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train,X_test = X.iloc[train_index],X.iloc[test_index]
        y_train,y_test = y.iloc[train_index],y.iloc[test_index]
        pipeline.fit(X_train,y_train.values.ravel())
        y_pred = pipeline.predict(X_test)
        f1_scr= f1_score(y_test,y_pred) 
        average_f1_score += f1_scr

    return {'params': hp, 'Average F1 scores': average_f1_score/5}        
        
    

In [162]:
best_params = performance_df.iloc[7]['params']
best_params

{'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}

In [175]:
# Testing F1 Score compute
logisitc= LogisticRegression()
logisitc.set_params(**best_params)
preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_column),
        ('num', StandardScaler(), numerical_column)
    ])

pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', logisitc)
    ])
pipeline.fit(X_train_imputed,y_train.values.ravel())
print(preprocessor.named_transformers_["cat"].get_feature_names_out())
y_pred = pipeline.predict(X_test_imputed)
f1_scr= f1_score(y_test,y_pred) 
print(f1_scr)


['Sex_0.0' 'Sex_1.0' 'ChestPainType_1.0' 'ChestPainType_2.0'
 'ChestPainType_3.0' 'ChestPainType_4.0' 'FBS_0.0' 'FBS_1.0' 'RestECG_0.0'
 'RestECG_1.0' 'RestECG_2.0' 'ExAng_0.0' 'ExAng_1.0' 'Slope_1.0'
 'Slope_2.0' 'Slope_3.0' 'Ca_0.0' 'Ca_0.5' 'Ca_1.0' 'Ca_2.0' 'Ca_3.0'
 'Thal_3.0' 'Thal_6.0' 'Thal_6.500000000000001' 'Thal_7.0']
0.7381974248927039
